In [21]:
# =========================
# 🔥 OPTIMIZED PIPELINE (RESNET50) — FIXED + TQDM
# =========================

import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
from torch.cuda.amp import autocast, GradScaler
import os
import shutil
from tqdm.auto import tqdm  # ✅ ADDED

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# =========================
# 🧹 REMOVE HIDDEN FOLDERS (FINAL FIX)
# =========================

def deep_clean(path):
    for root, dirs, files in os.walk(path):
        for d in dirs:
            if d.startswith('.'):
                shutil.rmtree(os.path.join(root, d), ignore_errors=True)

DATA_DIR = 'data'
train_dir = os.path.join(DATA_DIR, 'classification', 'train')
test_dir  = os.path.join(DATA_DIR, 'classification', 'test')

deep_clean(train_dir)
deep_clean(test_dir)

# =========================
# 📂 DATA PREPROCESSING (384 FINAL)
# =========================

train_transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(12),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)
])

test_transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)
])

def is_valid_file(path):
    valid_ext = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')
    if any(part.startswith('.') for part in path.split('/')):
        return False
    return path.lower().endswith(valid_ext)

train_data = datasets.ImageFolder(
    train_dir,
    transform=train_transform,
    is_valid_file=is_valid_file
)

test_data = datasets.ImageFolder(
    test_dir,
    transform=test_transform,
    is_valid_file=is_valid_file
)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_data, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

classes = train_data.classes
print("Classes:", classes)
print("Train size:", len(train_data))

# =========================
# 🧠 MODEL
# =========================

model = models.resnet50(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 4)
model = model.to(device)

# =========================
# ⚙️ HELPER
# =========================

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating", leave=False):  # ✅ ADDED
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

# =========================
# 🔒 STAGE 1: WARMUP
# =========================

for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
scaler = GradScaler()

print("\n🔥 Stage 1: Warmup")

for epoch in range(5):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/5", leave=False)  # ✅ ADDED

    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())  # ✅ ADDED

    acc = evaluate(model, test_loader)
    print(f"Epoch {epoch+1}, Loss: {total_loss:.3f}, Val Acc: {acc:.4f}")

# =========================
# 🔓 STAGE 2: FINE-TUNE + EARLY STOPPING
# =========================

for name, param in model.named_parameters():
    if "layer4" in name or "layer3" in name:
        param.requires_grad = True

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

best_acc = 0
patience = 8
counter = 0

print("\n🔥 Stage 2: Fine-tuning")

for epoch in range(25):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/30", leave=False)  # ✅ ADDED

    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())  # ✅ ADDED

    scheduler.step()
    acc = evaluate(model, test_loader)

    print(f"Epoch {epoch+1}, Loss: {total_loss:.3f}, Val Acc: {acc:.4f}")

    if acc > best_acc:
        best_acc = acc
        counter = 0
        torch.save(model.state_dict(), "best_model.pth")
    else:
        counter += 1

    if counter >= patience:
        print("⛔ Early stopping triggered")
        break

# =========================
# 📊 FINAL EVALUATION
# =========================

model.load_state_dict(torch.load("best_model.pth"))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Final Eval"):  # ✅ ADDED
        images = images.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
print("\n📊 Confusion Matrix:\n", cm)

report = classification_report(all_labels, all_preds, target_names=classes)
print("\n📊 Classification Report:\n", report)

accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
print("\n✅ Final Accuracy:", accuracy)

Classes: ['glioma', 'meningioma', 'no_tumor', 'pituitary']
Train size: 4944


/tmp/ipykernel_301840/4044403560.py:125: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()



🔥 Stage 1: Warmup


Epoch 1/5:   0%|          | 0/78 [00:00<?, ?it/s]/tmp/ipykernel_301840/4044403560.py:138: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1, Loss: 66.896, Val Acc: 0.7120


...

Epoch 23, Loss: 1.660, Val Acc: 0.9850


Epoch 24, Loss: 1.393, Val Acc: 0.9890


Epoch 25, Loss: 1.483, Val Acc: 0.9830


Final Eval: 100%|██████████| 16/16 [00:01<00:00,  8.26it/s]


📊 Confusion Matrix:
 [[250   4   0   0]
 [  2 301   2   1]
 [  0   0 140   0]
 [  0   2   0 298]]

📊 Classification Report:
               precision    recall  f1-score   support

      glioma       0.99      0.98      0.99       254
  meningioma       0.98      0.98      0.98       306
    no_tumor       0.99      1.00      0.99       140
   pituitary       1.00      0.99      0.99       300

    accuracy                           0.99      1000
   macro avg       0.99      0.99      0.99      1000
weighted avg       0.99      0.99      0.99      1000


✅ Final Accuracy: 0.989
